# Time Series Momentum, Backtested on Crypto
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nikshev/unstuck/blob/main/episodes/q02-time-series-momentum/q02_time_series_momentum.ipynb)

Time-series momentum as published by Moskowitz, Ooi and Pedersen, *Time Series Momentum*,
Journal of Financial Economics (2012): every month, go long an asset whose past-12-month return
is positive and short one whose return is negative, each sized to the same volatility.
We test it on every coin that had a Binance perpetual in January 2020 — long and short,
with fees and funding. Education, not financial advice.

Two differences from the paper, stated up front: it uses *excess* returns (minus the risk-free rate),
we use plain returns; it annualises volatility with 261 trading days, crypto trades all 365.

## 1. Install
freqtrade is the backtester; we call only a few of its methods and explain each one.

In [ ]:
!pip install -q freqtrade==2026.10

## 2. Helpers from the earlier episodes
`binance_vision` — raw Binance data with SHA-256 checks (built in q00); `quant_report` — freqtrade's verdict and hourly marked-to-market equity (built in q01).

In [ ]:
!curl -fsSLO https://raw.githubusercontent.com/nikshev/unstuck/main/episodes/quant-lib/binance_vision.py -O https://raw.githubusercontent.com/nikshev/unstuck/main/episodes/quant-lib/quant_report.py
from binance_vision import save_for_freqtrade, DATA
from quant_report import verdict, equity, drawdown
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 3. The universe: every perpetual Binance listed by January 2020
Chosen by listing date, not by who is still around today — so it includes EOS, which stopped trading in 2025. Uses `save_for_freqtrade`, `DATA` (section 2).

In [ ]:
coins = ['BTC', 'ETH', 'BCH', 'XRP', 'EOS', 'LTC', 'TRX', 'ETC', 'LINK', 'XLM', 'ADA']
for c in coins:
    save_for_freqtrade(c + "USDT", "1d", "2020-01", "2026-10")
close = pd.DataFrame({c: pd.read_feather(DATA / f"{c}_USDT_USDT-1d-futures.feather").set_index("date")["close"]
                      for c in coins})
print(close.apply(lambda s: f"{s.first_valid_index().date()} to {s.last_valid_index().date()}").to_string())

## 4. The signal: the sign of the past 12 months
Return over 365 days, $r_{12m} = P_t / P_{t-365} - 1$; long if it is positive, short if negative. Uses `close` (section 3).

In [ ]:
mom = close / close.shift(365) - 1
month_end = mom.index[(mom.index + pd.Timedelta(days=1)).day == 1]
signal = np.sign(mom.loc[month_end])
print("BTC, last six month-ends:")
print(pd.DataFrame({"12m return": mom.loc[month_end, "BTC"], "signal": signal["BTC"]}).tail(6).round(3).to_string())

## 5. The size: every coin gets the same risk
Ex-ante volatility $\sigma$ = exponentially weighted standard deviation of daily returns (centre of mass 60 days) × $\sqrt{365}$. Weight $= \dfrac{40\%}{\sigma} \cdot \dfrac{1}{S}$ of the account, S = the number of coins trading that day (the paper's $S_t$). Uses `close` (3), `month_end` (4).

In [ ]:
sigma = close.pct_change().ewm(com=60).std() * np.sqrt(365)
last = month_end[-1]
S = close.loc[last].notna().sum()
weight = 0.40 / sigma / S
print("coins trading on", last.date(), ":", S)
print(pd.DataFrame({"volatility": sigma.loc[last], "weight": weight.loc[last]}).round(3).to_string())

## 6. Backtest settings
**Cross** margin: eleven small positions share one wallet, so one coin doubling does not liquidate a short on its own. Leverage 1, 10,000 USDT, the strategy sizes every position.

In [ ]:
config = {
    "trading_mode": "futures", "margin_mode": "cross", "liquidation_buffer": 0.0,
    "stake_currency": "USDT", "stake_amount": "unlimited", "dry_run_wallet": 10000,
    "max_open_trades": 11, "timeframe": "1d", "dry_run": True, "fee": 0.0005,
    "entry_pricing": {"price_side": "same", "use_order_book": True, "order_book_top": 1},
    "exit_pricing": {"price_side": "same", "use_order_book": True, "order_book_top": 1},
    "exchange": {"name": "binance", "pair_whitelist": [c + "/USDT:USDT" for c in ['BTC', 'ETH', 'BCH', 'XRP', 'EOS', 'LTC', 'TRX', 'ETC', 'LINK', 'XLM', 'ADA']]},
    "pairlists": [{"method": "StaticPairList"}],
}
json.dump(config, open("config.json", "w"), indent=1)

## 7. Strategy file, part 1: the settings
12-month lookback, 40% target volatility, leverage 1. freqtrade insists on a stoploss: −99% is as far as it goes.

In [ ]:
%%writefile user_data/strategies/TSMOM.py
import numpy as np
import pandas as pd
from freqtrade.strategy import IStrategy

class TSMOM(IStrategy):
    LOOKBACK, TARGET, LEV, SHORTS = 365, 0.40, 1, True
    timeframe = "1d"
    can_short = True
    minimal_roi = {"0": 100}
    stoploss = -0.99
    position_adjustment_enable = True
    startup_candle_count = 366

## 8. Part 2: indicators
The same volatility and 12-month return as sections 4–5, and a flag on the last day of each month. `LOOKBACK` from section 7.

In [ ]:
%%writefile -a user_data/strategies/TSMOM.py

    def populate_indicators(self, df, metadata):
        df["sigma"] = df["close"].pct_change().ewm(com=60).std() * np.sqrt(365)
        df["mom"] = df["close"] / df["close"].shift(self.LOOKBACK) - 1
        df["month_end"] = (df["date"] + pd.Timedelta(days=1)).dt.day == 1
        return df

## 9. Part 3: entries and exits
On a month's last close: long if momentum is positive, short if negative; leave the position whose sign flipped. The order fills at the next open — the 1st. Columns from section 8, `SHORTS` from 7.

In [ ]:
%%writefile -a user_data/strategies/TSMOM.py

    def populate_entry_trend(self, df, metadata):
        df.loc[df["month_end"] & (df["mom"] > 0), "enter_long"] = 1
        if self.SHORTS:
            df.loc[df["month_end"] & (df["mom"] < 0), "enter_short"] = 1
        return df

    def populate_exit_trend(self, df, metadata):
        df.loc[df["month_end"] & (df["mom"] < 0), "exit_long"] = 1
        df.loc[df["month_end"] & (df["mom"] > 0), "exit_short"] = 1
        return df

## 10. Part 4: the size of a position
Weight from section 5 × the whole account = the position; margin = position / leverage. S = `current_whitelist()`, the pairs freqtrade actually trades. `TARGET`, `LEV` from section 7, `sigma` from 8.

In [ ]:
%%writefile -a user_data/strategies/TSMOM.py

    def leverage(self, pair, **kwargs):
        return self.LEV

    def target_margin(self, pair):
        df, _ = self.dp.get_analyzed_dataframe(pair, self.timeframe)
        weight = self.TARGET / float(df["sigma"].iloc[-1]) / len(self.dp.current_whitelist())
        return weight * self.wallets.get_total_stake_amount() / self.LEV

    def custom_stake_amount(self, pair, current_time, current_rate, proposed_stake, min_stake,
                            max_stake, leverage, entry_tag, side, **kwargs):
        return min(self.target_margin(pair), max_stake)

## 11. Part 5: rebalance once a month
On the 1st, bring every open position back to its target: buy the difference (positive) or sell it (negative); skip changes under 10% to save fees. `target_margin` from section 10.

In [ ]:
%%writefile -a user_data/strategies/TSMOM.py

    def adjust_trade_position(self, trade, current_time, current_rate, current_profit, min_stake,
                              max_stake, current_entry_rate, current_exit_rate,
                              current_entry_profit, current_exit_profit, **kwargs):
        if current_time.day != 1 or trade.has_open_orders or trade.open_date_utc.date() == current_time.date():
            return None
        target = self.target_margin(trade.pair)
        now = trade.amount * current_rate / trade.leverage
        change = target - now
        if abs(change) < 0.1 * target:
            return None
        return min(change, max_stake) if change > 0 else max(change, -0.999 * now)

## 12. Run it
From 2 January 2021 — the first day with twelve months of history. Uses `config.json` (6), `TSMOM` (7–11), `verdict`, `equity`, `drawdown` (section 2).

In [ ]:
!freqtrade backtesting --cache none -c config.json -s TSMOM --timerange 20210102-20261031 --export trades > run.log 2>&1
both, trades = verdict("TSMOM")
both["drawdown, marked to market"] = drawdown(equity(trades, start="2021-01-02"))

## 13. One coin is missing
freqtrade checks every pair against the markets Binance lists **today** — and drops the delisted ones. Our data has EOS; the backtest does not. Uses `run.log` (section 12), `close` (3).

In [ ]:
!grep -m1 "EOS" run.log
eos = close["EOS"].dropna()
print("EOS:", eos.index[0].date(), eos.iloc[0], "->", eos.index[-1].date(), eos.iloc[-1],
      f"({eos.iloc[-1] / eos.iloc[0] - 1:.0%})")

## 14. Where the money went
Profit by coin and by side. Uses `trades` (section 12).

In [ ]:
print(trades.groupby(["pair", "is_short"])["profit_abs"].sum().unstack().round(0).rename(columns={False: "long", True: "short"}))
print(trades["exit_reason"].value_counts().to_string())

## 15. Long or cash
The same rules without shorts: a negative signal means no position. Inherits sections 7–11. Uses `verdict`, `equity`, `drawdown` (2).

In [ ]:
%%writefile user_data/strategies/TSMOMLong.py
from TSMOM import TSMOM

class TSMOMLong(TSMOM):
    SHORTS = False

In [ ]:
!freqtrade backtesting --cache none -c config.json -s TSMOMLong --timerange 20210102-20261031 --export trades > run.log 2>&1
long_only, trades_long = verdict("TSMOMLong")
long_only["drawdown, marked to market"] = drawdown(equity(trades_long, start="2021-01-02"))

## 16. Against holding all ten
Buy-and-hold = the ten coins freqtrade traded, equal amounts, rebalanced daily, valued at the daily close. Uses `close` (3), `drawdown` (2), `trades`, `trades_long` (12, 15).

In [ ]:
traded = close.drop(columns="EOS").loc["2021-01-02":]
hold = 10_000 * (1 + traded.pct_change().mean(axis=1).fillna(0)).cumprod()
curves = {"buy and hold, 10 coins": hold,
          "TSMOM long and short": equity(trades, start="2021-01-02"),
          "TSMOM long or cash": equity(trades_long, start="2021-01-02")}
for name, curve in curves.items():
    print(f"{name:>23}: final {curve.iloc[-1]:>8.0f}  max drawdown {drawdown(curve):.0%}")
plt.figure(figsize=(11, 5))
for name, curve in curves.items():
    plt.plot(curve.index, curve, label=name)
plt.yscale("log"); plt.legend(); plt.title("10,000 USDT, January 2021 to October 2026"); plt.show()

## 17. Is it the 12 months?
Crypto moves faster than the futures the paper tested — try 1, 3 and 6 months. Each is `TSMOM` with one setting changed (section 7).

In [ ]:
%%writefile user_data/strategies/TSMOMFast.py
from TSMOM import TSMOM

class TSMOM1m(TSMOM):
    LOOKBACK = 30

class TSMOM3m(TSMOM):
    LOOKBACK = 90

class TSMOM6m(TSMOM):
    LOOKBACK = 180

In [ ]:
fast = {}
for name in ["TSMOM1m", "TSMOM3m", "TSMOM6m"]:
    !freqtrade backtesting --cache none -c config.json -s {name} --timerange 20210102-20261031 --export trades > run.log 2>&1
    fast[name], t = verdict(name)
    fast[name]["drawdown, marked to market"] = drawdown(equity(t, start="2021-01-02"))

## 18. Every run side by side
Uses `both` (12), `long_only` (15), `fast` (17).

In [ ]:
table = pd.DataFrame({"12m long+short": both, "12m long or cash": long_only, **fast}).round(3)
print(table.to_string())